# Scene harvest: quick analysis

Loads the Parquet dataset written by `scene-harvest run` and answers three questions:
how heavy are the scenes, which dependencies are missing, and how many joints drive each skinned vertex.

Point `SCENE_HARVEST_OUTPUT` at a real harvest folder. Without it, the notebook builds the small USD
sample library from the tests and harvests it, so it runs anywhere `usd-core` is installed.

In [ ]:
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join("..", "python")))
from scene_harvest import cli
from scene_harvest import writers

OUTPUT = os.environ.get("SCENE_HARVEST_OUTPUT", os.path.abspath(os.path.join("..", "build", "demo_harvest")))

if not os.path.isdir(os.path.join(OUTPUT, "dataset")):
    from scene_harvest.test import usd_samples

    scenes_root = os.path.abspath(os.path.join("..", "build", "demo_scenes"))
    usd_samples.build(scenes_root)
    cli.main(["run", scenes_root, "-o", OUTPUT, "-w", "2", "-b", "1"])

tables = {name: writers.read_table(OUTPUT, name) for name in ("scenes", "meshes", "dependencies", "skins")}
{name: len(frame) for name, frame in tables.items()}

## Polycount distribution

Triangles per scene, plus the heaviest meshes. In an ML dataset this is the first thing to check:
a long tail of huge scenes usually needs its own sampling strategy.

In [ ]:
scenes, meshes = tables["scenes"], tables["meshes"]
per_scene = (
    meshes.groupby("scene_id")
    .agg(meshes=("shape_path", "count"), triangles=("triangle_count", "sum"), vertices=("vertex_count", "sum"))
    .join(scenes.set_index("scene_id")["scene_path"].map(os.path.basename))
    .sort_values("triangles", ascending=False)
)
print(per_scene["triangles"].describe())
per_scene

In [ ]:
buckets = pd.cut(meshes["triangle_count"], bins=[0, 100, 1_000, 10_000, 100_000, float("inf")])
buckets.value_counts().sort_index().rename("meshes")

## Missing dependencies

Which kinds of files are missing, and which missing paths are shared by the most scenes.

In [ ]:
dependencies = tables["dependencies"]
missing = dependencies[~dependencies["exists"]]
summary = dependencies.groupby("kind")["exists"].agg(total="count", present="sum")
summary["missing"] = summary["total"] - summary["present"]
summary.sort_values("missing", ascending=False)

In [ ]:
missing.groupby("path")["scene_id"].nunique().rename("scenes").sort_values(ascending=False).head(10)

## Influences per skin

Max and mean influences per vertex. Realtime targets usually cap at 4, so anything above is flagged.

In [ ]:
skins = tables["skins"].copy()
skins["over_budget"] = skins["max_influences_per_vertex"] > 4
columns = ["skin", "geometry", "influence_count", "vertex_count",
           "max_influences_per_vertex", "mean_influences_per_vertex", "over_budget"]
skins[columns]